# Code Review Assistant

Reviews Python code for **bugs, security issues, style issues and optimization opportunities**.

Pipeline: **preprocessing (ast) → static analysis (pylint + bandit) → AI review (Qwen2.5-Coder-7B, with static results as context) → verification → merge duplicates**

**Before running:** Runtime → Change runtime type → **T4 GPU**, and upload `analyzer.py` using the Files panel on the left. Then Runtime → Run all.

## 1. Install libraries

In [ ]:
# Cell 1: Install
!pip install -q transformers accelerate bitsandbytes pylint bandit gradio

## 2. Load the AI model
Qwen2.5-Coder-7B-Instruct with 4-bit quantization, so it fits on the free T4 GPU (about 5.5 GB instead of 15 GB).

In [ ]:
# Cell 2: Load the 7B model
from transformers.utils import logging
logging.set_verbosity_error()          # hide library warnings, show only real errors

import torch
from transformers import pipeline, BitsAndBytesConfig

model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-Coder-7B-Instruct",
    device_map="auto",
    model_kwargs={"quantization_config": BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    )},
)
print("7B model loaded!")

## 3. Prompt for the AI reviewer
Defines the 4 categories, asks for JSON only, requires the exact code as `evidence`, and gives a few-shot example.

In [ ]:
# Cell 3: Prompt
system_prompt = """You are a strict Python code reviewer. Check every line for:
- bug: logic errors, wrong comparisons (> vs >=), crashes on empty input,
  division by zero, off-by-one errors, mutable default arguments
- security: eval, exec, shell commands, passwords in code
- style: unused imports, names that break PEP 8 (only if they really break it)
- optimization: slow or unnecessary code

Think about edge cases: empty lists, zero, None, boundary values.

Reply ONLY with a JSON list. Each item has: "line", "category", "message", "fix", "evidence".
"evidence" must be the exact code copied from that line.

Example input:
1: import sys
2: def first_item(values):
3:     return values[0]
4: def run(a):
5:     return exec(a)

Example output:
[{"line": 1, "category": "style", "message": "sys is imported but never used", "fix": "remove the import", "evidence": "import sys"},
 {"line": 3, "category": "bug", "message": "crashes with IndexError when values is empty", "fix": "check if values is empty first", "evidence": "return values[0]"},
 {"line": 5, "category": "security", "message": "exec runs any code the user gives", "fix": "avoid exec; parse input safely", "evidence": "return exec(a)"}]"""

## 4. AI review function
Sends line-numbered code to the model, together with the static tool findings as JSON context, and converts the answer into the common issue format.

In [ ]:
# Cell 4: AI review function (static tool results are given to the AI as context)
import json


def clean_json(text):
    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1:
        return []
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return []


def llm_review(code, static_issues=None, syntax_error=None):
    numbered = ""
    for i, line in enumerate(code.split("\n"), start=1):
        numbered += f"{i}: {line}\n"

    user_message = "Review this code:\n" + numbered

    # give the static tool results to the AI as structured context
    if static_issues:
        tool_findings = [{"line": s["line"], "category": s["category"], "tool": s["tool"], "message": s["message"]}
                         for s in static_issues]
        user_message += ("\nStatic analysis tools already found these issues (JSON):\n"
                         + json.dumps(tool_findings)
                         + "\nUse them as context. Explain or fix them if useful, "
                           "and focus on problems the tools missed, like logic errors and edge cases.\n")

    # tell the AI if the code is incomplete
    if syntax_error:
        user_message += (f"\nNote: the code has a syntax error on line {syntax_error['line']} "
                         f"({syntax_error['message']}). It may be incomplete. Review what you can.\n")

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    output = model(messages, max_new_tokens=500, do_sample=False)
    answer = output[0]["generated_text"][-1]["content"]

    found = []
    for item in clean_json(answer):
        found.append({
            "line": item.get("line", 0),
            "category": item.get("category", "bug") if item.get("category") in ["bug", "security", "style", "optimization"] else "bug",
            "message": item.get("message", ""),
            "fix": item.get("fix", ""),
            "evidence": item.get("evidence", ""),
            "tool": "llm",
        })
    return found

## 5. Full review pipeline
1. **Preprocess:** normalise the code and parse it with `ast`; syntax errors are reported, not fatal
2. **Static analysis:** pylint + bandit (from `analyzer.py`)
3. **AI review:** with static results as context; each AI answer is verified against the real code
4. **Merge:** issues with the same line and category are combined

In [ ]:
# Cell 5: Preprocessing, verification and the full review pipeline
import ast
from analyzer import run_pylint, run_bandit


def preprocess(code):
    # clean up the code and parse it
    code = code.replace("\r\n", "\n").replace("\t", "    ")   # same line endings, tabs to spaces
    if not code.endswith("\n"):
        code += "\n"

    info = {"code": code, "lines": code.split("\n"), "syntax_error": None, "functions": []}
    try:
        tree = ast.parse(code)                                   # turn the code into a syntax tree
        info["functions"] = [node.name for node in ast.walk(tree)
                             if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
    except SyntaxError as e:
        info["syntax_error"] = {"line": e.lineno or 1, "message": e.msg}
    return info


def is_real(issue, code_lines):
    line_no = issue["line"]
    if line_no < 1 or line_no > len(code_lines):
        return False

    evidence = issue["evidence"].replace(" ", "")
    if evidence == "":
        return True

    for n in [line_no - 1, line_no, line_no + 1]:
        if 1 <= n <= len(code_lines):
            if evidence in code_lines[n - 1].replace(" ", ""):
                issue["line"] = n          # correct the AI's line number
                return True
    return False


def review(code):
    # 1. preprocessing
    info = preprocess(code)
    code = info["code"]
    code_lines = info["lines"]

    # 2. static analysis
    with open("sample.py", "w") as f:
        f.write(code)
    static_issues = run_pylint("sample.py") + run_bandit("sample.py")

    if info["syntax_error"]:
        static_issues.append({
            "line": info["syntax_error"]["line"],
            "category": "bug",
            "message": "Syntax error: " + info["syntax_error"]["message"],
            "tool": "preprocess",
        })

    # 3. AI review, with static results as context, then verify each answer
    ai_issues = []
    removed = 0
    for issue in llm_review(code, static_issues, info["syntax_error"]):
        if is_real(issue, code_lines):
            ai_issues.append(issue)
        else:
            removed += 1
    print(f"Removed {removed} fake AI answer(s)")

    # 4. merge duplicates
    final = {}
    for issue in static_issues + ai_issues:
        key = (issue["line"], issue["category"])
        if key in final:
            if issue["tool"] not in final[key]["tools"]:
                final[key]["tools"].append(issue["tool"])
            if issue["tool"] != "llm" and issue["message"] not in final[key]["message"]:
                final[key]["message"] += " / " + issue["message"]
            if issue.get("fix"):
                final[key]["fix"] = issue["fix"]
        else:
            issue["tools"] = [issue["tool"]]
            final[key] = issue

    return sorted(final.values(), key=lambda x: x["line"])

## 6. Tests

In [ ]:
# Cell 6: Test with bad code (unused import, mutable default, eval)
code = """import os

def add(x, items=[]):
    items.append(x)
    return eval(input())
"""

for issue in review(code):
    print(issue["line"], issue["category"], issue["tools"], issue["message"])

In [ ]:
# Cell 7: Test the fake-answer check (hallucination filter)
code_lines = code.split("\n")

fake = {"line": 3, "category": "security", "message": "password hard-coded", "evidence": "password = '123'", "tool": "llm"}
real = {"line": 5, "category": "security", "message": "eval is unsafe", "evidence": "eval(input())", "tool": "llm"}

print("Fake answer kept?", is_real(fake, code_lines))
print("Real answer kept?", is_real(real, code_lines))

In [ ]:
# Cell 8: Test logic bugs (static tools cannot find these)
logic_code = """def is_adult(age):
    return age > 18


def average(numbers):
    return sum(numbers) / len(numbers)
"""

for issue in review(logic_code):
    print(issue["line"], issue["category"], issue["tools"], issue["message"])

In [ ]:
# Cell 9: Test incomplete / broken code (handled gracefully)
broken_code = """def parse(line):
    parts = line.split(",")
    if len(parts) > 2
        return eval(parts[0])
"""

info = preprocess(broken_code)
print("Syntax error found:", info["syntax_error"])

for issue in review(broken_code):
    print(issue["line"], issue["category"], issue["tools"], issue["message"])

## 7. Interactive mode: Gradio web page
Paste code, click Submit, and see the review. `share=True` creates a temporary public link for the demo.

In [ ]:
# Cell 10: Gradio web page
import gradio as gr


def review_ui(code_text):
    issues = review(code_text)
    if not issues:
        return "No issues found."

    text = ""
    for issue in issues:
        text += f"Line {issue['line']} | {issue['category'].upper()} | found by: {', '.join(issue['tools'])}\n"
        text += f"   Problem: {issue['message']}\n"
        if issue.get("fix"):
            text += f"   Fix: {issue['fix']}\n"
        text += "\n"
    return text


demo = gr.Interface(
    fn=review_ui,
    inputs=gr.Code(language="python", label="Paste your Python code"),
    outputs=gr.Textbox(label="Review result", lines=20),
    title="Code Review Assistant",
    description="Static analysis (pylint + bandit) combined with an open-source AI model (Qwen 7B).",
)

demo.launch(share=True)

## 8. Labelled test cases
12 snippets with known answers (line, category): bugs (including logic bugs), security, style, optimization, and 2 clean snippets to check for false alarms.

In [ ]:
# Cell 11: Test cases (code with known answers for evaluation)
test_cases = [
    {
        "name": "mutable default",
        "code": """def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags
""",
        "expected": [(1, "bug")],
    },
    {
        "name": "shell injection",
        "code": """import subprocess


def ping(host):
    return subprocess.call("ping " + host, shell=True)
""",
        "expected": [(5, "security")],
    },
    {
        "name": "clean code",
        "code": """def square(x):
    return x * x
""",
        "expected": [],
    },
    {
        "name": "empty average",
        "code": """def average(values):
    return sum(values) / len(values)
""",
        "expected": [(2, "bug")],
    },
    {
        "name": "eval input",
        "code": """def calculate(text):
    return eval(text)
""",
        "expected": [(2, "security")],
    },
    {
        "name": "hardcoded password",
        "code": """PASSWORD = "admin123"


def login(user):
    return user == "admin"
""",
        "expected": [(1, "security")],
    },
    {
        "name": "unused import",
        "code": """import sys


def greet(name):
    return "Hello " + name
""",
        "expected": [(1, "style")],
    },
    {
        "name": "bad function name",
        "code": """def CalculateTotal(a, b):
    return a + b
""",
        "expected": [(1, "style")],
    },
    {
        "name": "none comparison",
        "code": """def has_value(x):
    if x == None:
        return False
    return True
""",
        "expected": [(2, "style")],
    },
    {
        "name": "slow string",
        "code": """def join_words(words):
    result = ""
    for word in words:
        result += word
    return result
""",
        "expected": [(4, "optimization")],
    },
    {
        "name": "first item crash",
        "code": """def first(items):
    return items[0]
""",
        "expected": [(2, "bug")],
    },
    {
        "name": "clean code 2",
        "code": """def is_even(n):
    return n % 2 == 0
""",
        "expected": [],
    },
]

print(len(test_cases), "test cases ready")

In [ ]:
# Show each test case with line numbers (to check the expected answers)
for case in test_cases:
    print("=====", case["name"], "| expected:", case["expected"])
    for i, line in enumerate(case["code"].split("\n"), start=1):
        print(f"{i}: {line}")
    print()

## 9. Evaluation
Compares 3 modes on the test cases: **static** (pylint + bandit), **llm** (AI only, verified), **hybrid** (full pipeline).
A prediction is correct if the category matches and the line is within ±1.

In [ ]:
# Cell 12: Evaluation - precision, recall, F1 and time for static vs llm vs hybrid
import time
import importlib
import analyzer
importlib.reload(analyzer)                 # load the latest analyzer.py
from analyzer import run_pylint, run_bandit


def get_predictions(code, mode):
    if mode == "static":
        with open("sample.py", "w") as f:
            f.write(code)
        issues = run_pylint("sample.py") + run_bandit("sample.py")
    elif mode == "llm":
        code_lines = code.split("\n")
        issues = [i for i in llm_review(code) if is_real(i, code_lines)]
    else:
        issues = review(code)
    # keep only (line, category), with no duplicates
    return list({(i["line"], i["category"]) for i in issues})


def count_matches(predicted, expected):
    tp = 0
    remaining = list(expected)
    for p_line, p_cat in predicted:
        for e in remaining:
            if p_cat == e[1] and abs(p_line - e[0]) <= 1:   # same category, line within 1
                tp += 1
                remaining.remove(e)
                break
    fp = len(predicted) - tp
    fn = len(remaining)
    return tp, fp, fn


results = {}
for mode in ["static", "llm", "hybrid"]:
    total_tp = total_fp = total_fn = 0
    start = time.time()
    for case in test_cases:
        predicted = get_predictions(case["code"], mode)
        tp, fp, fn = count_matches(predicted, case["expected"])
        total_tp += tp
        total_fp += fp
        total_fn += fn
    seconds = time.time() - start

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    results[mode] = (precision, recall, f1, total_tp, total_fp, total_fn, seconds)

print(f"{'Mode':<8} {'Precision':>9} {'Recall':>7} {'F1':>5} {'TP':>4} {'FP':>4} {'FN':>4} {'Time(s)':>8}")
for mode, (p, r, f1, tp, fp, fn, s) in results.items():
    print(f"{mode:<8} {p:>9.2f} {r:>7.2f} {f1:>5.2f} {tp:>4} {fp:>4} {fn:>4} {s:>8.1f}")

In [ ]:
# Cell 13: Per-category evaluation (e.g. style accuracy compared to the PEP 8 style guide)
categories = ["bug", "security", "style", "optimization"]

for mode in ["static", "hybrid"]:
    print(f"\n{mode.upper()}")
    print(f"{'Category':<13} {'Precision':>9} {'Recall':>7} {'TP':>4} {'FP':>4} {'FN':>4}")
    counts = {c: [0, 0, 0] for c in categories}

    for case in test_cases:
        predicted = get_predictions(case["code"], mode)
        for c in categories:
            p = [x for x in predicted if x[1] == c]
            e = [x for x in case["expected"] if x[1] == c]
            tp, fp, fn = count_matches(p, e)
            counts[c][0] += tp
            counts[c][1] += fp
            counts[c][2] += fn

    for c, (tp, fp, fn) in counts.items():
        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        print(f"{c:<13} {precision:>9.2f} {recall:>7.2f} {tp:>4} {fp:>4} {fn:>4}")

In [ ]:
# Cell 14: Error analysis - what did hybrid get wrong?
for case in test_cases:
    predicted = get_predictions(case["code"], "hybrid")
    expected = case["expected"]
    extra = [p for p in predicted if not any(p[1] == e[1] and abs(p[0] - e[0]) <= 1 for e in expected)]
    missed = [e for e in expected if not any(p[1] == e[1] and abs(p[0] - e[0]) <= 1 for p in predicted)]
    if extra or missed:
        print(case["name"], "| extra:", extra, "| missed:", missed)

## 10. Batch mode (CI/CD)
Reviews every `.py` file in a folder, saves one Markdown report, and prints a CI status (FAIL if security issues are found).

In [ ]:
# Cell 15: Batch mode
import os


def review_folder(folder, report_file="review_report.md"):
    report = "# Code Review Report\n\n"
    total_security = 0

    for filename in sorted(os.listdir(folder)):
        if not filename.endswith(".py"):
            continue
        with open(os.path.join(folder, filename)) as f:
            code = f.read()

        issues = review(code)
        report += f"## {filename} ({len(issues)} issues)\n\n"
        for issue in issues:
            report += f"- Line {issue['line']} | {issue['category'].upper()} | {', '.join(issue['tools'])}: {issue['message']}\n"
            if issue["category"] == "security":
                total_security += 1
        report += "\n"

    with open(report_file, "w") as f:
        f.write(report)

    print(f"Report saved to {report_file}")
    # a CI pipeline could stop the build when security issues are found
    print("CI status:", "FAIL (security issues found)" if total_security else "PASS")


# demo: write 3 test cases as files, then review the whole folder
os.makedirs("samples", exist_ok=True)
for case in test_cases[:3]:
    with open(f"samples/{case['name'].replace(' ', '_')}.py", "w") as f:
        f.write(case["code"])

review_folder("samples")
print(open("review_report.md").read())